# V11 XAI Workflow

This notebook is the working XAI companion for the current best model family in the project.

It covers:

1. why each XAI method is chosen
2. which model layer or representation it explains
3. how to interpret the resulting plots
4. where the saved XAI figures live


## Research-Backed Methods

- Integrated Gradients: Sundararajan et al., *Axiomatic Attribution for Deep Networks*.
- Grad-CAM: Selvaraju et al., *Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization*.
- SHAP: Lundberg and Lee, *A Unified Approach to Interpreting Model Predictions*.
- WindowSHAP: time-series adaptation of Shapley explanations for sequential models.
- ECG-specific guidance: recent ECG XAI review and ECG heatmap evaluation papers.

Why these methods fit this project:

- `V9` and the deep branches in `V11` are convolution-based waveform models, so `Integrated Gradients` and `Grad-CAM` are appropriate.
- The handcrafted feature branch is feature-based, so `SHAP` is appropriate.
- `Occlusion sensitivity` is useful as a faithfulness check.
- `Fusion-weight analysis` is appropriate for `V11` because it is a late-fusion model.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path(r"C:\Users\Admin\Desktop\Projects\8th Sem\Codex")
REPORTS = ROOT / "reports"
ASSETS = REPORTS / "assets"
V9_PLOTS = ROOT.parent / "Preprocessed_Dataset" / "v4_beats" / "plots"
V11_SUMMARY = ROOT / "outputs" / "v11_triexpert_fusion" / "summary_all5.json"

summary = json.loads(V11_SUMMARY.read_text(encoding="utf-8"))
summary.keys()


## V11 Fusion-Level XAI

These are the new V11-specific XAI plots created from the validated 5-fold runs.


In [ ]:
display(Image(filename=str(ASSETS / "v11_xai_choice_counts.png")))
display(Image(filename=str(ASSETS / "v11_xai_delta_vs_v9.png")))
display(Image(filename=str(ASSETS / "v11_xai_weighted_expert_mix.png")))
pd.read_csv(ASSETS / "v11_xai_method_matrix.csv")


### How to read the V11 fusion plots

- `choice_counts`: shows how often each fusion strategy won on validation.
- `delta_vs_v9`: shows whether the full V11 system really improves over the deep V9 expert.
- `weighted_expert_mix`: shows which expert contributed most in weighted-grid folds.


## Existing Waveform and Feature XAI Plots

These come from the already-developed V9/XAI pipeline and remain relevant because `V11` still relies heavily on the V9 deep expert.


In [ ]:
xai_files = [
    "xai1_integrated_gradients_per_class.png",
    "xai1_ig_lead_heatmap.png",
    "xai2_gradcam_per_class.png",
    "xai2_gradcam_vs_ig_agreement.png",
    "xai4_occlusion_sensitivity.png",
    "xai5_cross_attention_weights.png",
    "shap_global_importance.png",
    "shap_summary_normal_non-cardiac.png",
    "shap_summary_structural heart disease.png",
    "shap_summary_arrhythmia & electrical.png",
]

for name in xai_files:
    path = V9_PLOTS / name
    display(Markdown(f"### {name}"))
    display(Image(filename=str(path)))


## Interpretation Guide

- Integrated Gradients:
  Positive attribution over a waveform region means that region pushed the model toward the predicted class.
- Grad-CAM:
  Hotter regions indicate the final convolutional representation is focusing there.
- SHAP:
  Large absolute contribution means a feature strongly changes the prediction.
- Occlusion:
  If masking a segment drops confidence sharply, that segment is important.
- Cross-attention / fusion:
  Higher weights mean the model relied more on rhythm or on a particular expert.


## Source Links

- [Integrated Gradients](https://arxiv.org/abs/1703.01365)
- [Grad-CAM](https://arxiv.org/abs/1610.02391)
- [SHAP](https://arxiv.org/abs/1705.07874)
- [WindowSHAP](https://arxiv.org/abs/2211.06507)
- [Evaluating gradient-based explanation methods for ECG using heatmaps](https://pmc.ncbi.nlm.nih.gov/articles/PMC11648713/)
- [Explainable artificial intelligence in electrocardiography: A systematic review](https://pmc.ncbi.nlm.nih.gov/articles/PMC13056371/)
